# Benchmark v1 vs v2 — Population Cap Fix

**Question:** Does fixing the benchmark's population caps change the conclusion?
**Answer:** Yes. VDP's benchmark advantage over digest2 shrinks substantially once MBA is
fairly represented and TNO/Trojan are no longer artificially inflated.

---

## What changed between v1 and v2

The original benchmark (`gen_benchmark_tracklets_s3m.py`) applied hard caps to each population
**before** propagation, keeping MBA at 200k and Trojan at 100k while leaving NEO and TNO
uncapped. This suppressed MBA ~69× below its true catalog share:

| Population | True S3M catalog | True share | v1 cap | v2 cap (proportional) |
|---|---:|---:|---:|---:|
| NEO     | 268,512   | 1.87%  | uncapped (all in) | 12,900 |
| MBA     | 13,883,375 | 96.54% | **200,000**       | **650,000** |
| TNO     | 48,682    | 0.34%  | uncapped (all in) | 1,600 |
| Trojan  | 179,883   | 1.25%  | **100,000**       | **6,000** |

v2 back-solves each cap from the population's own observed survival rate so that the **final scored counts** reflect the true catalog proportions.  
The original v1 file is kept intact — v2 is a fresh independent run.

---

> **Note on the `mag_bin_label` filter.**  
> VDP assigns a `mag_bin_label` only when the tracklet falls within the probability-map
> footprint.  Objects outside all map cells have `mag_bin_label = NaN` and `P_NEO_vdp = 0`.
> All ROC calculations below use only `mag_bin_label.notna()` rows — the subset that VDP
> actually scored.  digest2 always scores (no footprint requirement), so this filter only
> removes objects VDP could not place in a map cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import precision_recall_curve, roc_auc_score

def roc_xy(yy, s):
    """(completeness%, contamination%, bestF1, best_compl%, best_contam%)"""
    p, r, _ = precision_recall_curve(yy, s)
    f1 = np.divide(2*p*r, p+r, out=np.zeros_like(p), where=(p+r)>0)
    bi = int(np.argmax(f1[:-1]))
    return r*100, (1-p)*100, f1[bi], r[bi]*100, (1-p[bi])*100

POPS   = ['NEO', 'MBA', 'TNO', 'Trojans']
BINS   = [(0,20),(20,40),(40,70),(70,110),(110,141)]
BIN_LABS = ['0–20°','20–40°','40–70°','70–110°','110–141°']
C_VDP  = 'tab:blue'
C_D2   = 'tab:orange'

In [ ]:
# ── load and filter ───────────────────────────────────────────────────────────
paths = {
    'v1 (original caps)': 'outputs/phase2_benchmark_s3m/benchmark_comparison_s3m.parquet',
    'v2 (proportional caps)': 'docs/benchmark_comparison_s3m_v2.parquet',
}

benches = {}
for label, path in paths.items():
    raw = pd.read_parquet(path)
    df  = raw[raw.mag_bin_label.notna()].copy()
    df['absdlon'] = df.dlon_from_antisun_deg.abs()
    df['is_neo']  = (df.population == 'NEO')
    benches[label] = df
    n_raw = len(raw)
    n_flt = len(df)
    print(f'{label}:')
    print(f'  raw={n_raw:,}  scored={n_flt:,}  ({100*n_flt/n_raw:.1f}% within VDP footprint)')
    print(f'  pops: {dict(df.population.value_counts())}')
    print()

## Section 1 — Population Summary

In [ ]:
# ── population count / % table ────────────────────────────────────────────────
rows = []
for ver, df in benches.items():
    vc = df.population.value_counts()
    total = len(df)
    row = {'Version': ver, 'Total (VDP scored)': total}
    for p in POPS:
        n = int(vc.get(p, 0))
        row[p] = f'{n:,}  ({100*n/total:.1f}%)'
    rows.append(row)
pop_tbl = pd.DataFrame(rows).set_index('Version')
print('=== Population breakdown (VDP-scored subset) ===')
display(pop_tbl)

In [ ]:
# ── side-by-side stacked bar ──────────────────────────────────────────────────
pop_colors = {'NEO':'tab:red','MBA':'steelblue','TNO':'gold','Trojans':'mediumseagreen'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (ver, df) in zip(axes, benches.items()):
    vc = df.population.value_counts()
    total = len(df)
    bottom = 0.0
    for pop in POPS:
        frac = 100 * vc.get(pop, 0) / total
        bar = ax.bar(0, frac, bottom=bottom, color=pop_colors[pop], label=pop, width=0.4)
        if frac > 1:
            ax.text(0, bottom + frac/2, f'{pop}\n{frac:.1f}%',
                    ha='center', va='center', fontsize=9,
                    color='white' if pop == 'MBA' else 'black')
        bottom += frac
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(0, 100)
    ax.set_xticks([])
    ax.set_ylabel('% of VDP-scored tracklets')
    ax.set_title(ver, fontsize=9)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Benchmark population mix: original caps vs proportional caps', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/bench_pop_mix.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 2 — ROC: VDP vs digest2, v1 vs v2

In [ ]:
# ── 1×2 ROC panels, one per benchmark version ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

summary_rows = []
for ax, (ver, df) in zip(axes, benches.items()):
    yy   = df.is_neo.values
    cv, kv, fv, mcv, mkv = roc_xy(yy, df.P_NEO_vdp.values)
    cd, kd, fd, mcd, mkd = roc_xy(yy, df.P_NEO_d2.values)
    auv  = roc_auc_score(yy, df.P_NEO_vdp.values)
    aud  = roc_auc_score(yy, df.P_NEO_d2.values)

    ax.plot(cv, kv, lw=2.5, color=C_VDP, label=f'VDP   F1={fv:.3f}  AUC={auv:.3f}')
    ax.plot(cd, kd, lw=2.5, color=C_D2,  ls='--', label=f'digest2  F1={fd:.3f}  AUC={aud:.3f}')
    ax.scatter([mcv], [mkv], color=C_VDP, s=70, zorder=5)
    ax.scatter([mcd], [mkd], color=C_D2,  s=70, marker='s', zorder=5)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_title(f'{ver}\nN_NEO={int(yy.sum()):,}  /  N_total={len(df):,}', fontsize=9)
    ax.set_xlabel('NEO completeness (%)')

    winner = 'VDP' if fv > fd else ('digest2' if fd > fv else 'tied')
    summary_rows.append({
        'Version': ver,
        'N_NEO': int(yy.sum()), 'NEO %': f'{100*yy.mean():.1f}',
        'VDP F1': round(fv,3), 'VDP compl%': round(mcv,1), 'VDP contam%': round(mkv,1), 'VDP AUC': round(auv,3),
        'd2 F1':  round(fd,3), 'd2 compl%':  round(mcd,1), 'd2 contam%':  round(mkd,1), 'd2 AUC':  round(aud,3),
        'Winner': winner,
    })

axes[0].set_ylabel('Contamination (%)')
fig.suptitle('Benchmark v1 vs v2 — VDP vs digest2 (full sky, VDP-scored subset)', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/bench_roc_v1_v2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── summary table ─────────────────────────────────────────────────────────────
print('=== Full-sky ROC summary ===')
display(pd.DataFrame(summary_rows).set_index('Version'))

In [ ]:
# ── overlay: VDP curves for both versions, then d2 curves ─────────────────────
fig, (ax_v, ax_d) = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

ls_map = {'v1 (original caps)': '-', 'v2 (proportional caps)': '--'}
alpha_map = {'v1 (original caps)': 1.0, 'v2 (proportional caps)': 0.85}

for ver, df in benches.items():
    yy = df.is_neo.values
    cv, kv, fv, mcv, mkv = roc_xy(yy, df.P_NEO_vdp.values)
    cd, kd, fd, mcd, mkd = roc_xy(yy, df.P_NEO_d2.values)
    ls = ls_map[ver]; al = alpha_map[ver]

    ax_v.plot(cv, kv, lw=2.5, color=C_VDP, ls=ls, alpha=al, label=f'{ver}  F1={fv:.3f}')
    ax_v.scatter([mcv], [mkv], color=C_VDP, s=65, zorder=5)
    ax_d.plot(cd, kd, lw=2.5, color=C_D2,  ls=ls, alpha=al, label=f'{ver}  F1={fd:.3f}')
    ax_d.scatter([mcd], [mkd], color=C_D2,  s=65, marker='s', zorder=5)

for ax, title in [(ax_v,'VDP'), (ax_d,'digest2')]:
    ax.set_xlim(0,100); ax.set_ylim(0,100)
    ax.set_xlabel('NEO completeness (%)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(f'{title} — v1 vs v2 overlay')
ax_v.set_ylabel('Contamination (%)')

fig.suptitle('Effect of population-cap fix on classifier ROC curves', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/bench_roc_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3 — Per-Direction-Bin ROC

The population-mix effect is not uniform across sky direction.  
v1's TNO inflation is concentrated at the antisun (0–20°), where TNOs cluster at opposition.
Trojan inflation dominates the 40–110° range.  
After the cap fix (v2), MBA dominates everywhere and the playing field is level.

In [ ]:
# ── per-bin population % table ────────────────────────────────────────────────
bin_rows = []
for ver, df in benches.items():
    for blab, (lo, hi) in zip(BIN_LABS, BINS):
        m = (df.absdlon >= lo) & (df.absdlon < hi)
        vc = df[m].population.value_counts()
        total = int(m.sum())
        row = {'Version': ver.split()[0], 'Bin': blab, 'N': total}
        for p in POPS:
            n = int(vc.get(p, 0))
            row[p] = f'{100*n/total:.1f}%' if total else '—'
        bin_rows.append(row)
print('=== Population % by direction bin ===')
display(pd.DataFrame(bin_rows).set_index(['Version','Bin']))

In [ ]:
# ── 2×5 ROC grid: v1 (top) vs v2 (bottom), five direction bins ───────────────
fig, axes = plt.subplots(2, 5, figsize=(17, 7), sharex=True, sharey=True)

f1_bin_rows = []
for row_i, (ver, df) in enumerate(benches.items()):
    yy_all = df.is_neo.values
    for col_i, (blab, (lo, hi)) in enumerate(zip(BIN_LABS, BINS)):
        ax = axes[row_i, col_i]
        m  = ((df.absdlon >= lo) & (df.absdlon < hi)).values
        yy = yy_all[m]

        if yy.sum() < 5:
            ax.text(0.5, 0.5, f'N_NEO={yy.sum()}\n(too few)', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8)
            continue

        cv, kv, fv, mcv, mkv = roc_xy(yy, df.P_NEO_vdp.values[m])
        cd, kd, fd, mcd, mkd = roc_xy(yy, df.P_NEO_d2.values[m])

        ax.plot(cv, kv, lw=1.8, color=C_VDP,  label=f'VDP {fv:.2f}')
        ax.plot(cd, kd, lw=1.8, color=C_D2, ls='--', label=f'd2 {fd:.2f}')
        ax.scatter([mcv],[mkv], color=C_VDP, s=45, zorder=5)
        ax.scatter([mcd],[mkd], color=C_D2,  s=45, marker='s', zorder=5)

        # shade who wins
        win_color = C_VDP if fv > fd else C_D2
        ax.set_facecolor((*plt.matplotlib.colors.to_rgb(win_color), 0.05))

        ax.set_xlim(0,100); ax.set_ylim(0,100)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=6.5, loc='upper left')

        n_neo = int(yy.sum())
        if row_i == 0:
            ax.set_title(f'{blab}\nN_NEO={n_neo:,}', fontsize=8)
        else:
            ax.set_title(f'N_NEO={n_neo:,}', fontsize=8)

        f1_bin_rows.append({'Version': ver.split()[0], 'Bin': blab,
                            'N_NEO': n_neo, 'VDP F1': round(fv,3), 'd2 F1': round(fd,3),
                            'Winner': 'VDP' if fv > fd else 'd2'})

    short = ver.split()[0]  # v1 or v2
    axes[row_i, 0].set_ylabel(f'{ver}\nContamination (%)', fontsize=7)

for ax in axes[1, :]:
    ax.set_xlabel('Completeness (%)', fontsize=8)

fig.suptitle('ROC by direction bin: benchmark v1 (original caps, top) vs v2 (proportional, bottom)\n'
             'Background tint = winner (blue=VDP, orange=digest2)', fontsize=10)
plt.tight_layout()
plt.savefig('Figures/bench_roc_by_bin.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── F1 per bin summary table ──────────────────────────────────────────────────
f1_bin = pd.DataFrame(f1_bin_rows)
print('=== Per-bin F1 summary ===')
display(f1_bin.set_index(['Version', 'Bin']))

# pivot for easy reading
piv_vdp = f1_bin.pivot(index='Bin', columns='Version', values='VDP F1')
piv_d2  = f1_bin.pivot(index='Bin', columns='Version', values='d2 F1')
print('\n=== VDP F1 by bin ===')
display(piv_vdp)
print('\n=== digest2 F1 by bin ===')
display(piv_d2)

## Section 4 — Interpretation

### What the cap fix reveals

**v1 (original, artifically inflated TNO/Trojan):**
- 0–20° (antisun): TNO = 38.6% of the bin → d2 confuses TNOs with NEOs (short-arc orbit fitting
  cannot distinguish a ≈stationary TNO from an elongated-orbit NEO); VDP correctly scores
  TNOs near the origin (slow velocity → low P(NEO)).  VDP wins by +0.15 F1.
- 70–141°: Trojans dominate; d2 also struggles with Trojans at these elongations.
- Net: VDP F1=0.787 >> d2 F1=0.667.

**v2 (proportional, MBA-realistic):**
- MBA now 96% of non-NEO (true catalog share). MBA moves ~0.2 deg/day at the antisun —
  close to the NEO velocity range, making classification genuinely harder for both.
- d2's orbit-fitting advantage on MBA (well-constrained short arcs) now matters more.
- VDP F1=0.627, d2 F1=0.677 — d2 slightly ahead across most bins.
- VDP still competitive at 70–141° (F1 0.60–0.73), where its 2D velocity structure
  separates some NEOs from MBA even away from the antisun.

### Implication for the paper

The benchmark v1 result (VDP wins) was driven by the population-cap artifact, not by
a genuine algorithmic advantage in a realistic population.  The honest benchmark comparison
is v2, which agrees with the Sorcha direction: d2 is slightly ahead full-sky, VDP is
competitive or ahead at the antisun (0–20°) even with realistic population mix.

The TNO contamination proof (Section 15, `sorcha_v5_normalisation_s3m.ipynb`) already
showed this causally for the 0–20° bin.  v2 confirms it holds globally.